In [20]:
!pwd

/mnt/c/Users/Asus/jupyter_work/POS


In [25]:
!head mypos-ver.3.0.txt

ဒီ/adj ဆေး/n က/ppm ၁၀၀/num ရာခိုင်နှုန်း/n ဆေးဘက်ဝင်/adj အပင်/n များ/part မှ/ppm ဖော်စပ်/v ထား/part တာ/part ဖြစ်/v တယ်/ppm ။/punc
အသစ်/n ဝယ်/v ထား/part တဲ့/part ဆွယ်တာ/n က/ppm အသီး/n ထ/v နေ/part ပါ/part ပေါ့/part ။/punc
မ/part ကျန်းမာ/v လျှင်/conj နတ်/n|ဆရာ/n ထံ/ppm မေးမြန်း/v ၍/conj သက်ဆိုင်ရာ/n နတ်/n တို့/part အား/ppm ပူဇော်ပသ/v ရ/part သည်/ppm ။/punc
ပေဟိုင်/n|ဥယျာဉ်/n ။/punc
နဝမ/adj အိပ်မက်/n ကောသလ/n|မင်း/n|အိပ်မက်/n ၉/num နက်ရှိုင်း/adj ကျယ်ဝန်း/adj သော/part ရေကန်/n ကြီး/adj တစ်/tn ခု/part တွင်/ppm သတ္တဝါ/n တို့/part ဆင်း/v ၍/conj ရေသောက်/v ကြ/part ၏/ppm ။/punc
အပြင်ပန်း/n ကြည့်/v ရင်/conj ခက်/adj သလို/part ထင်/v ရ/part ပေမယ့်/conj တကယ့်/adj လက်တွေ့/n အခြေအနေ/n က/ppm တော့/part အဲဒီ/pron လို/ppm မ/part ဟုတ်/v ပါ/part ဘူး/part ။/punc
8/fw bit/fw ပုံရိပ်/n တစ်/tn ခု/part သည်/ppm 256/fw color/fw သို့မဟုတ်/conj gray/fw scale/fw များ/part ကို/ppm အထောက်အကူ/n ပြု/v သည်/ppm ။/punc
ကိုရီးယား/n ဝတ်စုံ/n မှာ/ppm ပန်း/n ဒီဇိုင်း/n နဲ့/conj အဝါရောင်/n က/ppm လိုက်ဖက်/v လိမ့်/part မယ်/part ထင်/v 

In [38]:
%%writefile reformat_mypos.py
import sys

def reformat_dataset(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    formatted_sentences = []

    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        tokens = line.split()
        sentence_lines = []
        
        for token in tokens:
            if '/' in token:
                word, tag = token.rsplit('/', 1)
                sentence_lines.append(f"{word}\t{tag}")
        
        if sentence_lines:
            formatted_sentences.append("\n".join(sentence_lines))

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("\n\n".join(formatted_sentences) + "\n\n")

    print(f"Done! Formatted {len(formatted_sentences)} sentences into '{output_file}'")

if __name__ == '__main__':
    reformat_dataset("mypos-ver.3.0.txt", "all_formatted.pos.txt")

Overwriting reformat_mypos.py


### Split Dataset for train and test 

In [39]:
%%writefile split_data.py
import sys

def split_dataset(input_file, train_file, test_file, train_ratio=0.8):
    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read().strip()

    sentences = content.split('\n\n')

    split_idx = int(len(sentences) * train_ratio)
    train_sentences = sentences[:split_idx]
    test_sentences = sentences[split_idx:]

    with open(train_file, 'w', encoding='utf-8') as f:
        f.write("\n\n".join(train_sentences) + "\n\n")

    with open(test_file, 'w', encoding='utf-8') as f:
        f.write("\n\n".join(test_sentences) + "\n\n")

    print(f"Total Sentences: {len(sentences)}")
    print(f"Training set ({int(train_ratio*100)}%): {len(train_sentences)} sentences -> saved to '{train_file}'")
    print(f"Testing set ({int((1-train_ratio)*100)}%):  {len(test_sentences)} sentences -> saved to '{test_file}'")

if __name__ == '__main__':
    split_dataset("all_formatted.pos.txt", "train.pos.txt", "test.pos.txt")

Writing split_data.py


In [40]:
!python3 reformat_mypos.py
!python3 split_data.py

Done! Formatted 43196 sentences into 'all_formatted.pos.txt'
Total Sentences: 43196
Training set (80%): 34556 sentences -> saved to 'train.pos.txt'
Testing set (19%):  8640 sentences -> saved to 'test.pos.txt'


In [42]:
!head -n 20 train.pos.txt

ဒီ	adj
ဆေး	n
က	ppm
၁၀၀	num
ရာခိုင်နှုန်း	n
ဆေးဘက်ဝင်	adj
အပင်	n
များ	part
မှ	ppm
ဖော်စပ်	v
ထား	part
တာ	part
ဖြစ်	v
တယ်	ppm
။	punc

အသစ်	n
ဝယ်	v
ထား	part
တဲ့	part


In [3]:
!head -n 20 test.pos.txt 

ယွမ်	n
ကို	ppm
ဘတ်ငွေ	n
လဲ	v
ချင်	part
လို့	part
ပါ	part
။	punc

ခင်ဗျား	pron
ဘယ်	adj
အချိန်	n
လာ	v
ယူ	v
ချင်	part
လဲ	part
။	punc

ဒီ	pron
မှာ	ppm


### Feature Extraction via chunking.py

In [43]:
!cat train.pos.txt | python3 /mnt/c/Users/Asus/jupyter_work/tool/crfsuite/example/chunking.py > train.pos.crfsuite.txt

In [45]:
!head -n 20 train.pos.crfsuite.txt

adj	w[0]=ဒီ	w[1]=ဆေး	w[2]=က	w[0]|w[1]=ဒီ|ဆေး	__BOS__
n	w[-1]=ဒီ	w[0]=ဆေး	w[1]=က	w[2]=၁၀၀	w[-1]|w[0]=ဒီ|ဆေး	w[0]|w[1]=ဆေး|က
ppm	w[-2]=ဒီ	w[-1]=ဆေး	w[0]=က	w[1]=၁၀၀	w[2]=ရာခိုင်နှုန်း	w[-1]|w[0]=ဆေး|က	w[0]|w[1]=က|၁၀၀
num	w[-2]=ဆေး	w[-1]=က	w[0]=၁၀၀	w[1]=ရာခိုင်နှုန်း	w[2]=ဆေးဘက်ဝင်	w[-1]|w[0]=က|၁၀၀	w[0]|w[1]=၁၀၀|ရာခိုင်နှုန်း
n	w[-2]=က	w[-1]=၁၀၀	w[0]=ရာခိုင်နှုန်း	w[1]=ဆေးဘက်ဝင်	w[2]=အပင်	w[-1]|w[0]=၁၀၀|ရာခိုင်နှုန်း	w[0]|w[1]=ရာခိုင်နှုန်း|ဆေးဘက်ဝင်
adj	w[-2]=၁၀၀	w[-1]=ရာခိုင်နှုန်း	w[0]=ဆေးဘက်ဝင်	w[1]=အပင်	w[2]=များ	w[-1]|w[0]=ရာခိုင်နှုန်း|ဆေးဘက်ဝင်	w[0]|w[1]=ဆေးဘက်ဝင်|အပင်
n	w[-2]=ရာခိုင်နှုန်း	w[-1]=ဆေးဘက်ဝင်	w[0]=အပင်	w[1]=များ	w[2]=မှ	w[-1]|w[0]=ဆေးဘက်ဝင်|အပင်	w[0]|w[1]=အပင်|များ
part	w[-2]=ဆေးဘက်ဝင်	w[-1]=အပင်	w[0]=များ	w[1]=မှ	w[2]=ဖော်စပ်	w[-1]|w[0]=အပင်|များ	w[0]|w[1]=များ|မှ
ppm	w[-2]=အပင်	w[-1]=များ	w[0]=မှ	w[1]=ဖော်စပ်	w[2]=ထား	w[-1]|w[0]=များ|မှ	w[0]|w[1]=မှ|ဖော်စပ်
v	w[-2]=များ	w[-1]=မှ	w[0]=ဖော်စပ်	w[1]=ထား	w[2]=တာ	w[-1]|w[0]=မှ|ဖော်စပ်	w[0]|w[1]=ဖော်စပ်|ထား
part	w[-2]=မှ	

In [44]:
!cat test.pos.txt | python3 /mnt/c/Users/Asus/jupyter_work/tool/crfsuite/example/chunking.py > test.pos.crfsuite.txt

In [46]:
!head -n 20 test.pos.crfsuite.txt

n	w[0]=ယွမ်	w[1]=ကို	w[2]=ဘတ်ငွေ	w[0]|w[1]=ယွမ်|ကို	__BOS__
ppm	w[-1]=ယွမ်	w[0]=ကို	w[1]=ဘတ်ငွေ	w[2]=လဲ	w[-1]|w[0]=ယွမ်|ကို	w[0]|w[1]=ကို|ဘတ်ငွေ
n	w[-2]=ယွမ်	w[-1]=ကို	w[0]=ဘတ်ငွေ	w[1]=လဲ	w[2]=ချင်	w[-1]|w[0]=ကို|ဘတ်ငွေ	w[0]|w[1]=ဘတ်ငွေ|လဲ
v	w[-2]=ကို	w[-1]=ဘတ်ငွေ	w[0]=လဲ	w[1]=ချင်	w[2]=လို့	w[-1]|w[0]=ဘတ်ငွေ|လဲ	w[0]|w[1]=လဲ|ချင်
part	w[-2]=ဘတ်ငွေ	w[-1]=လဲ	w[0]=ချင်	w[1]=လို့	w[2]=ပါ	w[-1]|w[0]=လဲ|ချင်	w[0]|w[1]=ချင်|လို့
part	w[-2]=လဲ	w[-1]=ချင်	w[0]=လို့	w[1]=ပါ	w[2]=။	w[-1]|w[0]=ချင်|လို့	w[0]|w[1]=လို့|ပါ
part	w[-2]=ချင်	w[-1]=လို့	w[0]=ပါ	w[1]=။	w[-1]|w[0]=လို့|ပါ	w[0]|w[1]=ပါ|။
punc	w[-2]=လို့	w[-1]=ပါ	w[0]=။	w[-1]|w[0]=ပါ|။	__EOS__

pron	w[0]=ခင်ဗျား	w[1]=ဘယ်	w[2]=အချိန်	w[0]|w[1]=ခင်ဗျား|ဘယ်	__BOS__
adj	w[-1]=ခင်ဗျား	w[0]=ဘယ်	w[1]=အချိန်	w[2]=လာ	w[-1]|w[0]=ခင်ဗျား|ဘယ်	w[0]|w[1]=ဘယ်|အချိန်
n	w[-2]=ခင်ဗျား	w[-1]=ဘယ်	w[0]=အချိန်	w[1]=လာ	w[2]=ယူ	w[-1]|w[0]=ဘယ်|အချိန်	w[0]|w[1]=အချိန်|လာ
v	w[-2]=ဘယ်	w[-1]=အချိန်	w[0]=လာ	w[1]=ယူ	w[2]=ချင်	w[-1]|w[0]=အချိန်|လာ	w[0]|w[1]=လာ|ယူ
v	w[-2]=

### Train with CRF model

In [52]:
!time /mnt/c/Users/Asus/jupyter_work/tool/crfsuite/frontend/crfsuite learn -m pos_tagger.model train.pos.crfsuite.txt

CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

Start time of the training: 2026-07-30T21:59:54Z

Reading the data set(s)
[1] train.pos.crfsuite.txt
0....1....2....3....4....5....6....7....8....9....10
Number of instances: 34557
Seconds required: 2.537

Statistics the data set(s)
Number of data sets (groups): 1
Number of instances: 34556
Number of items: 457401
Number of attributes: 473411
Number of labels: 15

Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 0
0....1....2....3....4....5....6....7....8....9....10
Number of features: 584912
Seconds required: 0.953

L-BFGS optimization
c1: 0.000000
c2: 1.000000
num_memories: 6
max_iterations: 2147483647
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

***** Iteration #1 *****
Loss: 786133.848506
Feature norm: 5.000000
Error norm: 61937.358206
Active features: 584912
Line search trials: 2
Line search step: 0.000045
Se

***** Iteration #36 *****
Loss: 128491.603133
Feature norm: 148.919692
Error norm: 1705.812883
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.414

***** Iteration #37 *****
Loss: 126886.558758
Feature norm: 152.801793
Error norm: 2113.613037
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.391

***** Iteration #38 *****
Loss: 125049.387749
Feature norm: 158.905806
Error norm: 2632.493476
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.409

***** Iteration #39 *****
Loss: 123170.105330
Feature norm: 160.802460
Error norm: 1903.659859
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.363

***** Iteration #40 *****
Loss: 121287.634828
Feature norm: 161.077958
Error norm: 1769.471450
Active features: 584912
Line search trials: 1
Line search

***** Iteration #75 *****
Loss: 97060.599337
Feature norm: 177.773956
Error norm: 660.576101
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.341

***** Iteration #76 *****
Loss: 96847.327025
Feature norm: 177.797119
Error norm: 1021.532468
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.342

***** Iteration #77 *****
Loss: 96643.992708
Feature norm: 177.819555
Error norm: 809.617785
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.339

***** Iteration #78 *****
Loss: 96484.419755
Feature norm: 177.810099
Error norm: 627.076100
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.344

***** Iteration #79 *****
Loss: 96308.608048
Feature norm: 177.800209
Error norm: 658.884073
Active features: 584912
Line search trials: 1
Line search step: 1.

***** Iteration #114 *****
Loss: 93655.553320
Feature norm: 180.473679
Error norm: 259.265034
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.356

***** Iteration #115 *****
Loss: 93617.020885
Feature norm: 180.685990
Error norm: 333.696324
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.394

***** Iteration #116 *****
Loss: 93595.242873
Feature norm: 180.874306
Error norm: 383.610670
Active features: 584912
Line search trials: 2
Line search step: 0.433612
Seconds required for this iteration: 0.695

***** Iteration #117 *****
Loss: 93567.959228
Feature norm: 181.075171
Error norm: 231.268274
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.384

***** Iteration #118 *****
Loss: 93545.640801
Feature norm: 181.252853
Error norm: 204.848347
Active features: 584912
Line search trials: 1
Line search step

***** Iteration #153 *****
Loss: 93231.662466
Feature norm: 185.693779
Error norm: 83.862296
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.359

***** Iteration #154 *****
Loss: 93229.210064
Feature norm: 185.779177
Error norm: 62.979982
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.353

***** Iteration #155 *****
Loss: 93227.890733
Feature norm: 185.853404
Error norm: 70.455593
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.363

***** Iteration #156 *****
Loss: 93225.425169
Feature norm: 186.054844
Error norm: 161.942312
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.355

***** Iteration #157 *****
Loss: 93222.639341
Feature norm: 186.158030
Error norm: 93.387026
Active features: 584912
Line search trials: 1
Line search step: 1.

***** Iteration #193 *****
Loss: 93193.124150
Feature norm: 186.829574
Error norm: 22.718671
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.356

***** Iteration #194 *****
Loss: 93192.909953
Feature norm: 186.830385
Error norm: 21.343843
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.349

***** Iteration #195 *****
Loss: 93192.665855
Feature norm: 186.830578
Error norm: 34.816018
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.355

***** Iteration #196 *****
Loss: 93192.439005
Feature norm: 186.829956
Error norm: 26.698338
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.367

***** Iteration #197 *****
Loss: 93192.244035
Feature norm: 186.828732
Error norm: 21.571239
Active features: 584912
Line search trials: 1
Line search step: 1.0

In [53]:
!/mnt/c/Users/Asus/jupyter_work/tool/crfsuite/frontend/crfsuite learn -e2 train.pos.crfsuite.txt test.pos.crfsuite.txt

CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

Start time of the training: 2026-07-30T22:01:47Z

Reading the data set(s)
[1] train.pos.crfsuite.txt
0....1....2....3....4....5....6....7....8....9....10
Number of instances: 34557
Seconds required: 2.689
[2] test.pos.crfsuite.txt
0....1....2....3....4....5....6....7....8....9....10
Number of instances: 8641
Seconds required: 0.414

Statistics the data set(s)
Number of data sets (groups): 2
Number of instances: 43196
Number of items: 537232
Number of attributes: 507034
Number of labels: 15

Holdout group: 2

Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 0
0....1....2....3....4....5....6....7....8....9....10
Number of features: 584912
Seconds required: 0.880

L-BFGS optimization
c1: 0.000000
c2: 1.000000
num_memories: 6
max_iterations: 2147483647
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

***** Iteration #1 **

***** Iteration #8 *****
Loss: 317411.300080
Feature norm: 35.687003
Error norm: 17641.547826
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.384
Performance by label (#match, #model, #ref) (precision, recall, F1):
    adj: (1179, 1654, 2694) (0.7128, 0.4376, 0.5423)
    n: (9711, 15446, 13016) (0.6287, 0.7461, 0.6824)
    ppm: (10815, 12961, 11661) (0.8344, 0.9275, 0.8785)
    num: (176, 1347, 575) (0.1307, 0.3061, 0.1831)
    part: (18943, 21465, 20726) (0.8825, 0.9140, 0.8980)
    v: (8980, 11401, 11289) (0.7877, 0.7955, 0.7915)
    punc: (9621, 9725, 9660) (0.9893, 0.9960, 0.9926)
    conj: (590, 974, 1146) (0.6057, 0.5148, 0.5566)
    tn: (907, 962, 1075) (0.9428, 0.8437, 0.8905)
    pron: (3356, 3561, 5442) (0.9424, 0.6167, 0.7455)
    fw: (0, 0, 560) (0.0000, 0.0000, 0.0000)
    adv: (272, 335, 1898) (0.8119, 0.1433, 0.2436)
    int: (0, 0, 57) (0.0000, 0.0000, 0.0000)
    abb: (0, 0, 19) (0.0000, 0.0000, 0.0000)
  

***** Iteration #15 *****
Loss: 207690.606984
Feature norm: 75.437117
Error norm: 6985.374137
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.371
Performance by label (#match, #model, #ref) (precision, recall, F1):
    adj: (1701, 1951, 2694) (0.8719, 0.6314, 0.7324)
    n: (11569, 15222, 13016) (0.7600, 0.8888, 0.8194)
    ppm: (11175, 11556, 11661) (0.9670, 0.9583, 0.9627)
    num: (255, 367, 575) (0.6948, 0.4435, 0.5414)
    part: (19673, 21267, 20726) (0.9250, 0.9492, 0.9370)
    v: (9775, 11538, 11289) (0.8472, 0.8659, 0.8564)
    punc: (9626, 9722, 9660) (0.9901, 0.9965, 0.9933)
    conj: (636, 844, 1146) (0.7536, 0.5550, 0.6392)
    tn: (972, 983, 1075) (0.9888, 0.9042, 0.9446)
    pron: (4991, 5095, 5442) (0.9796, 0.9171, 0.9473)
    fw: (7, 61, 560) (0.1148, 0.0125, 0.0225)
    adv: (934, 1218, 1898) (0.7668, 0.4921, 0.5995)
    int: (6, 7, 57) (0.8571, 0.1053, 0.1875)
    abb: (0, 0, 19) (0.0000, 0.0000, 0.0000)


***** Iteration #22 *****
Loss: 163421.204845
Feature norm: 104.097874
Error norm: 7379.636881
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.399
Performance by label (#match, #model, #ref) (precision, recall, F1):
    adj: (1836, 2078, 2694) (0.8835, 0.6815, 0.7695)
    n: (12155, 15386, 13016) (0.7900, 0.9339, 0.8559)
    ppm: (11442, 11693, 11661) (0.9785, 0.9812, 0.9799)
    num: (278, 295, 575) (0.9424, 0.4835, 0.6391)
    part: (19707, 20567, 20726) (0.9582, 0.9508, 0.9545)
    v: (10212, 11642, 11289) (0.8772, 0.9046, 0.8907)
    punc: (9627, 9723, 9660) (0.9901, 0.9966, 0.9933)
    conj: (693, 783, 1146) (0.8851, 0.6047, 0.7185)
    tn: (1015, 1027, 1075) (0.9883, 0.9442, 0.9657)
    pron: (5217, 5268, 5442) (0.9903, 0.9587, 0.9742)
    fw: (3, 33, 560) (0.0909, 0.0054, 0.0101)
    adv: (1199, 1330, 1898) (0.9015, 0.6317, 0.7429)
    int: (6, 6, 57) (1.0000, 0.1053, 0.1905)
    abb: (0, 0, 19) (0.0000, 0.0000, 0.0

***** Iteration #29 *****
Loss: 142498.454733
Feature norm: 130.218855
Error norm: 2982.317935
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.377
Performance by label (#match, #model, #ref) (precision, recall, F1):
    adj: (1944, 2150, 2694) (0.9042, 0.7216, 0.8026)
    n: (12126, 14460, 13016) (0.8386, 0.9316, 0.8827)
    ppm: (11396, 11499, 11661) (0.9910, 0.9773, 0.9841)
    num: (356, 374, 575) (0.9519, 0.6191, 0.7503)
    part: (19953, 20805, 20726) (0.9590, 0.9627, 0.9609)
    v: (10377, 11720, 11289) (0.8854, 0.9192, 0.9020)
    punc: (9627, 9708, 9660) (0.9917, 0.9966, 0.9941)
    conj: (900, 1111, 1146) (0.8101, 0.7853, 0.7975)
    tn: (1032, 1047, 1075) (0.9857, 0.9600, 0.9727)
    pron: (5300, 5338, 5442) (0.9929, 0.9739, 0.9833)
    fw: (2, 28, 560) (0.0714, 0.0036, 0.0068)
    adv: (1398, 1554, 1898) (0.8996, 0.7366, 0.8100)
    int: (13, 37, 57) (0.3514, 0.2281, 0.2766)
    abb: (0, 0, 19) (0.0000, 0.0000, 

...

***** Iteration #183 *****
Loss: 93196.461886
Feature norm: 186.782851
Error norm: 57.840014
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.498
Performance by label (#match, #model, #ref) (precision, recall, F1):
    adj: (2217, 2375, 2694) (0.9335, 0.8229, 0.8747)
    n: (12588, 14129, 13016) (0.8909, 0.9671, 0.9275)
    ppm: (11531, 11643, 11661) (0.9904, 0.9889, 0.9896)
    num: (482, 506, 575) (0.9526, 0.8383, 0.8918)
    part: (20193, 20694, 20726) (0.9758, 0.9743, 0.9750)
    v: (10676, 11443, 11289) (0.9330, 0.9457, 0.9393)
    punc: (9645, 9645, 9660) (1.0000, 0.9984, 0.9992)
    conj: (975, 1102, 1146) (0.8848, 0.8508, 0.8674)
    tn: (1054, 1067, 1075) (0.9878, 0.9805, 0.9841)
    pron: (5386, 5432, 5442) (0.9915, 0.9897, 0.9906)
    fw: (39, 52, 560) (0.7500, 0.0696, 0.1275)
    adv: (1619, 1701, 1898) (0.9518, 0.8530, 0.8997)
    int: (28, 30, 57) (0.9333, 0.4912, 0.6437)
    abb: (0, 0, 19) (0.0000, 0.0000, 0

***** Iteration #190 *****
Loss: 93193.954794
Feature norm: 186.819821
Error norm: 28.427984
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.484
Performance by label (#match, #model, #ref) (precision, recall, F1):
    adj: (2217, 2375, 2694) (0.9335, 0.8229, 0.8747)
    n: (12591, 14131, 13016) (0.8910, 0.9673, 0.9276)
    ppm: (11530, 11644, 11661) (0.9902, 0.9888, 0.9895)
    num: (482, 506, 575) (0.9526, 0.8383, 0.8918)
    part: (20192, 20693, 20726) (0.9758, 0.9742, 0.9750)
    v: (10675, 11440, 11289) (0.9331, 0.9456, 0.9393)
    punc: (9645, 9645, 9660) (1.0000, 0.9984, 0.9992)
    conj: (974, 1100, 1146) (0.8855, 0.8499, 0.8673)
    tn: (1054, 1067, 1075) (0.9878, 0.9805, 0.9841)
    pron: (5386, 5433, 5442) (0.9913, 0.9897, 0.9905)
    fw: (39, 52, 560) (0.7500, 0.0696, 0.1275)
    adv: (1619, 1702, 1898) (0.9512, 0.8530, 0.8994)
    int: (29, 31, 57) (0.9355, 0.5088, 0.6591)
    abb: (0, 0, 19) (0.0000, 0.0000, 0

***** Iteration #197 *****
Loss: 93192.244035
Feature norm: 186.828732
Error norm: 21.571239
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.433
Performance by label (#match, #model, #ref) (precision, recall, F1):
    adj: (2215, 2373, 2694) (0.9334, 0.8222, 0.8743)
    n: (12593, 14137, 13016) (0.8908, 0.9675, 0.9276)
    ppm: (11530, 11642, 11661) (0.9904, 0.9888, 0.9896)
    num: (482, 506, 575) (0.9526, 0.8383, 0.8918)
    part: (20193, 20693, 20726) (0.9758, 0.9743, 0.9751)
    v: (10677, 11442, 11289) (0.9331, 0.9458, 0.9394)
    punc: (9645, 9645, 9660) (1.0000, 0.9984, 0.9992)
    conj: (973, 1099, 1146) (0.8854, 0.8490, 0.8668)
    tn: (1054, 1067, 1075) (0.9878, 0.9805, 0.9841)
    pron: (5386, 5433, 5442) (0.9913, 0.9897, 0.9905)
    fw: (39, 52, 560) (0.7500, 0.0696, 0.1275)
    adv: (1618, 1700, 1898) (0.9518, 0.8525, 0.8994)
    int: (29, 31, 57) (0.9355, 0.5088, 0.6591)
    abb: (0, 0, 19) (0.0000, 0.0000, 0

***** Iteration #204 *****
Loss: 93191.180913
Feature norm: 186.822060
Error norm: 12.297348
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.442
Performance by label (#match, #model, #ref) (precision, recall, F1):
    adj: (2219, 2377, 2694) (0.9335, 0.8237, 0.8752)
    n: (12594, 14137, 13016) (0.8909, 0.9676, 0.9276)
    ppm: (11530, 11642, 11661) (0.9904, 0.9888, 0.9896)
    num: (482, 506, 575) (0.9526, 0.8383, 0.8918)
    part: (20191, 20690, 20726) (0.9759, 0.9742, 0.9750)
    v: (10677, 11441, 11289) (0.9332, 0.9458, 0.9395)
    punc: (9645, 9645, 9660) (1.0000, 0.9984, 0.9992)
    conj: (974, 1101, 1146) (0.8847, 0.8499, 0.8669)
    tn: (1054, 1067, 1075) (0.9878, 0.9805, 0.9841)
    pron: (5386, 5433, 5442) (0.9913, 0.9897, 0.9905)
    fw: (39, 52, 560) (0.7500, 0.0696, 0.1275)
    adv: (1618, 1698, 1898) (0.9529, 0.8525, 0.8999)
    int: (29, 31, 57) (0.9355, 0.5088, 0.6591)
    abb: (0, 0, 19) (0.0000, 0.0000, 0

***** Iteration #211 *****
Loss: 93190.485490
Feature norm: 186.814230
Error norm: 26.239073
Active features: 584912
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.442
Performance by label (#match, #model, #ref) (precision, recall, F1):
    adj: (2220, 2377, 2694) (0.9340, 0.8241, 0.8756)
    n: (12590, 14132, 13016) (0.8909, 0.9673, 0.9275)
    ppm: (11530, 11642, 11661) (0.9904, 0.9888, 0.9896)
    num: (482, 506, 575) (0.9526, 0.8383, 0.8918)
    part: (20191, 20690, 20726) (0.9759, 0.9742, 0.9750)
    v: (10678, 11447, 11289) (0.9328, 0.9459, 0.9393)
    punc: (9645, 9645, 9660) (1.0000, 0.9984, 0.9992)
    conj: (975, 1101, 1146) (0.8856, 0.8508, 0.8678)
    tn: (1054, 1067, 1075) (0.9878, 0.9805, 0.9841)
    pron: (5386, 5433, 5442) (0.9913, 0.9897, 0.9905)
    fw: (39, 52, 560) (0.7500, 0.0696, 0.1275)
    adv: (1618, 1697, 1898) (0.9534, 0.8525, 0.9001)
    int: (29, 31, 57) (0.9355, 0.5088, 0.6591)
    abb: (0, 0, 19) (0.0000, 0.0000, 0

In [2]:
!/mnt/c/Users/Asus/jupyter_work/tool/crfsuite/frontend/crfsuite tag -m pos_tagger.model test.pos.crfsuite.txt | head -n 20

n
ppm
n
v
part
part
part
punc

pron
adj
n
v
v
part
part
punc

pron
ppm


In [60]:
import subprocess

cmd = [
    "/mnt/c/Users/Asus/jupyter_work/tool/crfsuite/frontend/crfsuite",
    "tag",
    "-m", "pos_tagger.model",
    "test.pos.crfsuite.txt"
]

result = subprocess.run(cmd, stdout=subprocess.PIPE, universal_newlines=True)
predicted_tags = result.stdout.strip().split('\n')

words = []
with open("test.pos.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            parts = line.split('\t')
            words.append(parts[0])

# 3. Print side-by-side
print(f"{'WORD':<20} --> {'PREDICTED POS TAG'}")
print("-" * 40)

for word, tag in zip(words[:30], predicted_tags[:30]):
    print(f"{word:<20} --> {tag}")

WORD                 --> PREDICTED POS TAG
----------------------------------------
ယွမ်                 --> n
ကို                  --> ppm
ဘတ်ငွေ               --> n
လဲ                   --> v
ချင်                 --> part
လို့                 --> part
ပါ                   --> part
။                    --> punc
ခင်ဗျား              --> 
ဘယ်                  --> pron
အချိန်               --> adj
လာ                   --> n
ယူ                   --> v
ချင်                 --> v
လဲ                   --> part
။                    --> part
ဒီ                   --> punc
မှာ                  --> 
ဓာတ်ပုံ              --> pron
ရိုက်                --> ppm
လို့                 --> n
ရ                    --> v
မလား                 --> part
။                    --> v
ကျွန်တော်            --> part
သိ                   --> punc
ပါရစေ                --> 
။                    --> pron
ခင်ဗျား              --> v
ဘယ်                  --> ppm


### Testing with new sentences 

In [12]:
cat test_script.txt

ကျွန်မ ကော်ဖီ သောက် ချင် တယ် ။
သူ သည် ကျောင်းသား တစ် ယောက် ပါ ။

In [19]:
input_file_path = "test_script.txt"

with open(input_file_path, "r", encoding="utf-8") as f_in, open("formatted_input.txt", "w", encoding="utf-8") as f_out:
    for line in f_in:
        words = line.strip().split()
        if words:
            #used '?' as placeholder tag
            formatted_line = "\n".join([f"{word}\t?" for word in words])
            f_out.write(formatted_line + "\n\n")

print("Formatted input file successfully!")

Formatted input file successfully!


In [20]:
!cat formatted_input.txt | python3 /mnt/c/Users/Asus/jupyter_work/tool/crfsuite/example/chunking.py > input_features.crfsuite.txt

In [21]:
!head -n 20 input_features.crfsuite.txt

?	w[0]=ကျွန်မ	w[1]=ကော်ဖီ	w[2]=သောက်	w[0]|w[1]=ကျွန်မ|ကော်ဖီ	__BOS__
?	w[-1]=ကျွန်မ	w[0]=ကော်ဖီ	w[1]=သောက်	w[2]=ချင်	w[-1]|w[0]=ကျွန်မ|ကော်ဖီ	w[0]|w[1]=ကော်ဖီ|သောက်
?	w[-2]=ကျွန်မ	w[-1]=ကော်ဖီ	w[0]=သောက်	w[1]=ချင်	w[2]=တယ်	w[-1]|w[0]=ကော်ဖီ|သောက်	w[0]|w[1]=သောက်|ချင်
?	w[-2]=ကော်ဖီ	w[-1]=သောက်	w[0]=ချင်	w[1]=တယ်	w[2]=။	w[-1]|w[0]=သောက်|ချင်	w[0]|w[1]=ချင်|တယ်
?	w[-2]=သောက်	w[-1]=ချင်	w[0]=တယ်	w[1]=။	w[-1]|w[0]=ချင်|တယ်	w[0]|w[1]=တယ်|။
?	w[-2]=ချင်	w[-1]=တယ်	w[0]=။	w[-1]|w[0]=တယ်|။	__EOS__

?	w[0]=သူ	w[1]=သည်	w[2]=ကျောင်းသား	w[0]|w[1]=သူ|သည်	__BOS__
?	w[-1]=သူ	w[0]=သည်	w[1]=ကျောင်းသား	w[2]=တစ်	w[-1]|w[0]=သူ|သည်	w[0]|w[1]=သည်|ကျောင်းသား
?	w[-2]=သူ	w[-1]=သည်	w[0]=ကျောင်းသား	w[1]=တစ်	w[2]=ယောက်	w[-1]|w[0]=သည်|ကျောင်းသား	w[0]|w[1]=ကျောင်းသား|တစ်
?	w[-2]=သည်	w[-1]=ကျောင်းသား	w[0]=တစ်	w[1]=ယောက်	w[2]=ပါ	w[-1]|w[0]=ကျောင်းသား|တစ်	w[0]|w[1]=တစ်|ယောက်
?	w[-2]=ကျောင်းသား	w[-1]=တစ်	w[0]=ယောက်	w[1]=ပါ	w[2]=။	w[-1]|w[0]=တစ်|ယောက်	w[0]|w[1]=ယောက်|ပါ
?	w[-2]=တစ်	w[-1]=ယောက်	w[0]=ပါ	w[1]=။	w[-1]|w[0]=ယ

In [24]:
!/mnt/c/Users/Asus/jupyter_work/tool/crfsuite/frontend/crfsuite tag -m pos_tagger.model input_features.crfsuite.txt > input_predictions.txt

In [26]:
!head -n 20 input_predictions.txt

pron
n
v
part
ppm
punc

pron
ppm
n
tn
part
part
punc



In [29]:
with open("test_script.txt", "r", encoding="utf-8") as f_words, open("input_predictions.txt", "r", encoding="utf-8") as f_tags:
    words = [w for line in f_words for w in line.strip().split() if line.strip()]
    tags = [t.strip() for t in f_tags if t.strip()]

print(f"{'WORD':<20} | {'PREDICTED POS TAG'}")
print("-" * 40)
for word, tag in zip(words, tags):
    print(f"{word:<20} --> {tag}")

WORD                 | PREDICTED POS TAG
----------------------------------------
ကျွန်မ               --> pron
ကော်ဖီ               --> n
သောက်                --> v
ချင်                 --> part
တယ်                  --> ppm
။                    --> punc
သူ                   --> pron
သည်                  --> ppm
ကျောင်းသား           --> n
တစ်                  --> tn
ယောက်                --> part
ပါ                   --> part
။                    --> punc
